# Session 9 — Assignment 9: Loss harness and t+2 head
**Hari | ERA V5**

Run all cells in Google Colab. The notebook prints tensor shapes, literal token alignment, valid counts, losses and CPU peak RSS. Uses a small character vocabulary and deterministic initialization.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import math, random, subprocess, sys
torch.set_num_threads(2); random.seed(9);torch.manual_seed(9)
docs=['a cat sat.','a dog ran.','a cat ran.','a dog sat.']
vocab=['<pad>']+sorted(set(''.join(docs)))
stoi={x:i for i,x in enumerate(vocab)};itos={i:x for x,i in stoi.items()}
PAD=0;V=len(vocab);D=32
enc=lambda s:[stoi[c] for c in s]
dec=lambda ids:[itos[int(i)] for i in ids]
seq=[enc(d) for d in docs]; T=max(map(len,seq))+3
tokens=torch.tensor([s+[PAD]*(T-len(s)) for s in seq])
B,T=tokens.shape
print('vocab:',vocab,'V:',V)
print('tokens',tuple(tokens.shape),'[batch,time]')
print('inputs',tuple(tokens[:,:-1].shape),'[batch,time-1]')
print('targets',tuple(tokens[:,1:].shape),'[batch,time-1]')
print('LITERAL TOKEN SHIFT:')
for t,(x,y) in enumerate(zip(dec(tokens[0,:-1]),dec(tokens[0,1:]))):
 print(t,repr(x),'->',repr(y))
assert dec(tokens[0,:-1])[0]=='a' and dec(tokens[0,1:])[0]==' '

In [ ]:
class TinyLM(nn.Module):
 def __init__(self,tied=False):
  super().__init__()
  self.emb=nn.Embedding(V,D)
  self.context=nn.Linear(D,D)
  self.head1=nn.Linear(D,V,bias=False)
  if tied:self.head1.weight=self.emb.weight
  self.head2=nn.Linear(D,V,bias=False)
 def forward(self,x):
  e=self.emb(x); prior=torch.cat([torch.zeros_like(e[:,:1]),e[:,:-1]],1)
  h=torch.tanh(self.context(e+0.5*prior))
  return h,self.head1(h),self.head2(h)
m=TinyLM()
hidden,logits,logits2=m(tokens)
print('embedding',tuple(m.emb.weight.shape),'[vocab,hidden]')
print('hidden',tuple(hidden.shape),'[batch,time,hidden]')
print('head weights',tuple(m.head1.weight.shape),'[vocab,hidden]')
print('logits',tuple(logits.shape),'[batch,time,vocab]')
print('shifted logits',tuple(logits[:,:-1].shape),'[batch,time-1,vocab]')
print('flat logits',tuple(logits[:,:-1].reshape(-1,V).shape),'[batch*(time-1),vocab]')
print('flat targets',tuple(tokens[:,1:].reshape(-1).shape),'[batch*(time-1)]')
valid=tokens[:,1:]!=PAD
raw=F.cross_entropy(logits[:,:-1].reshape(-1,V),tokens[:,1:].reshape(-1))
masked=F.cross_entropy(logits[:,:-1].reshape(-1,V),tokens[:,1:].reshape(-1),ignore_index=PAD)
print('PAD: before',valid.numel(),'after',int(valid.sum()),'CE',raw.item(),'->',masked.item())

In [ ]:
a,b=seq[:2];packed=torch.tensor([a+b]);cut=len(a)
_,lp,_=m(packed)
terms=F.cross_entropy(lp[:,:-1].reshape(-1,V),packed[:,1:].reshape(-1),reduction='none')
keep=torch.ones_like(terms,dtype=torch.bool);keep[cut-1]=False
boundary_before=terms.mean();boundary_after=terms[keep].mean()
print('packed',tuple(packed.shape),'[batch,time]')
print('invalid transition:',repr(itos[a[-1]]),'->',repr(itos[b[0]]))
print('terms:',len(terms),'->',int(keep.sum()),'loss',boundary_before.item(),'->',boundary_after.item())
ppl=math.exp(masked.item());print('init PPL',ppl,'vocabulary',V,'ratio',ppl/V)
tied=TinyLM(True);untied=TinyLM(False)
count=lambda x:sum(p.numel() for p in x.parameters())
print('tied',count(tied),'untied',count(untied),'saved',count(untied)-count(tied))
assert count(untied)-count(tied)==V*D and 0.5*V<ppl<2*V

## CPU memory comparison
Two fresh subprocesses measure their own peak resident set size (RSS); this includes runtime overhead. The comparison is forward-only CE, not a CUDA or training-backward benchmark. Both modes use N=1024, D=64, V=8192, chunk size=64.

In [ ]:
worker = """
import torch,torch.nn.functional as F,resource,sys
torch.set_num_threads(2);torch.manual_seed(123)
N,D,V,C=1024,64,8192,64
x=torch.randn(N,D);w=torch.randn(V,D);y=torch.randint(V,(N,))
with torch.no_grad():
 if sys.argv[1]=='ordinary':
  loss=F.cross_entropy(x@w.T,y)
 else:
  total=torch.zeros(())
  for i in range(0,N,C):
   total+=F.cross_entropy(x[i:i+C]@w.T,y[i:i+C],reduction='sum')
  loss=total/N
rss=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
mib=rss/(1024*1024) if sys.platform=='darwin' else rss/1024
print(f'{loss.item()},{mib}')
"""
def measure(mode):
 out=subprocess.check_output([sys.executable,'-c',worker,mode],text=True).strip().splitlines()[-1]
 return tuple(map(float,out.split(',')))
lo,mo=measure('ordinary');lc,mc=measure('chunked')
print('ordinary loss / peak RSS MiB',lo,mo)
print('chunked loss / peak RSS MiB',lc,mc)
print('ordinary / chunked RSS ratio',mo/mc,'loss difference',abs(lo-lc))
assert abs(lo-lc)<1e-4

## Two output heads
Head 1 targets t+1; head 2 targets t+2. Targets are shifted forward, not backward. Equal-weight summed objective; ignore PAD independently for both.

In [ ]:
def losses(model,x):
 _,a,b=model(x)
 l1=F.cross_entropy(a[:,:-1].reshape(-1,V),x[:,1:].reshape(-1),ignore_index=PAD)
 l2=F.cross_entropy(b[:,:-2].reshape(-1,V),x[:,2:].reshape(-1),ignore_index=PAD)
 return l1,l2
torch.manual_seed(9)
train=TinyLM();optim=torch.optim.AdamW(train.parameters(),lr=.025,weight_decay=0)
print('t+1 string pairs',list(zip(dec(tokens[0,:-1]),dec(tokens[0,1:]))))
print('t+2 string pairs',list(zip(dec(tokens[0,:-2]),dec(tokens[0,2:]))))
history=[]
for step in range(151):
 optim.zero_grad();l1,l2=losses(train,tokens)
 if step%25==0:
  history.append((step,l1.item(),l2.item(),(l1+l2).item()))
  print(f'step={step:3d} t+1={l1.item():.6f} t+2={l2.item():.6f} sum={(l1+l2).item():.6f}')
 if step<150:(l1+l2).backward();optim.step()
assert history[-1][1]<history[0][1] and history[-1][2]<history[0][2]
print('t+2 starts similar, ends slightly above t+1 on this toy corpus; no held-out evaluation.')

## Limitations
A packed loss boundary mask eliminates one invalid supervised transition, but a real self-attention transformer also requires a block-diagonal causal attention mask to prevent cross-document leakage. The backbone here is only a causal current/previous token contextualizer. Memory results vary by CPU environment; run all cells for local measurements.